<a href="https://colab.research.google.com/github/filipchudzynski/stock-market-non-gaussianity-analyzer_v2/blob/main/log_energy_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! git clone https://github.com/filipchudzynski/stock-market-non-gaussianity-analyzer_v2.git

Cloning into 'stock-market-non-gaussianity-analyzer_v2'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 147 (delta 58), reused 73 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 23.82 MiB | 6.32 MiB/s, done.
Resolving deltas: 100% (58/58), done.


In [4]:
import sys
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append("/content/stock-market-non-gaussianity-analyzer_v2/log_energy_epjst_package/")
sys.path.append("/content/stock-market-non-gaussianity-analyzer_v2/log_energy_epjst_package/log_energy")
sys.path.append("/content/stock-market-non-gaussianity-analyzer_v2/log_energy_epjst_package/models")
sys.path.append("/content/stock-market-non-gaussianity-analyzer_v2/log_energy_epjst_package/tests")

from models.white_noise import white_noise
from models.brownian_motion import brownian_motion
from log_energy.operators import increment_operator
from log_energy.energy import log_energy_field, sliding_baseline,local_energy
from log_energy.mi_knn import mi_knn
from log_energy.intermittency import intermittency_variance
from log_energy.covariance import log_energy_covariance

In [34]:
N=5000
s=64
kappa=10
max_lag=200

def log_field_energy_computations(signal, s, kappa):
  window = max(int(kappa * s), 5)
  E = local_energy(signal)
  baseline = sliding_baseline(E, window)
  log_field = log_energy_field(signal,s,kappa)
  log_field_var = intermittency_variance(log_field)
  return E, baseline, log_field, log_field_var

wn = white_noise(N)
bm = brownian_motion(N)
for signal in [wn,bm]:
  E, baseline,log_field,log_field_var = log_field_energy_computations(signal, s, kappa)
  cov = log_energy_covariance(log_field,max_lag)


  increments = increment_operator(signal, s)
  E_inc, baseline_inc, log_field_increments, log_field_var_inc = log_field_energy_computations(increments, s,kappa)
  cov_inc = log_energy_covariance(log_field_increments,max_lag)


  mkr_size = 3
  opacity = 0.7
  marker=dict(line=dict(width=2))
  fig1 = make_subplots(rows=1, cols=3,
                         subplot_titles=["signal, energy, baseline","increments, energy, baseline", f"log-energy field [var: {log_field_var:2.3f}, {log_field_var_inc:2.3f}(inc)]"])
  fig1.add_trace(go.Scatter(y=signal,opacity=opacity, mode="lines",  marker=marker, name="signal"), row=1, col=1)
  fig1.add_trace(go.Scatter(y=E,opacity=opacity, mode="lines",   marker=marker,name="E"), row=1, col=1)
  fig1.add_trace(go.Scatter(y=baseline, mode="lines", name="baseline"), row=1, col=1)
  fig1.add_trace(go.Scatter(y=increments,opacity=opacity, mode="lines",   marker=marker,name="increments"), row=1, col=2)
  fig1.add_trace(go.Scatter(y=E_inc,opacity=opacity, mode="lines",  marker=marker, name="E_inc"), row=1, col=2)
  fig1.add_trace(go.Scatter(y=baseline_inc, mode="lines", name="baseline_inc"), row=1, col=2)
  fig1.add_trace(go.Scatter(y=log_field,opacity=opacity, mode="lines",  marker=marker, name="log energy field"), row=1, col=3)
  fig1.add_trace(go.Scatter(y=log_field_increments,opacity=opacity, mode="lines",  marker=marker, name="log energy field(inc)"), row=1, col=3)

  fig1.update_layout(height=600, title="signal, local energy, baseline, log energy field")
  fig1.show()

  fig2 = go.Figure()
  fig2.add_trace(go.Scatter(y=cov,mode="lines",name="Covariance"))
  fig2.add_trace(go.Scatter(y=cov_inc,mode="lines",name="Covariance increments"))
  fig2.show()



In [ ]:
from scipy.stats import kurtosis
from scipy.spatial import cKDTree
from scipy.special import digamma

# =========================
# Validation + comparison
# =========================

def validate_false_detection(N=8000, s=64, max_lag=200):

    # -------- Generate signals --------
    wn = white_noise(N)
    bm = brownian_motion(N)

    wn_inc = increment_operator(wn, s)
    bm_inc = increment_operator(bm, s)

    # -------- Naive methods (false detection) --------
    # 1) Kurtosis of increments
    wn_kurt = kurtosis(wn_inc, fisher=True, bias=False)
    bm_kurt = kurtosis(bm_inc, fisher=True, bias=False)

    # 2) Raw variance of increments
    wn_var_raw = np.var(wn_inc)
    bm_var_raw = np.var(bm_inc)

    # 3) Raw MI between successive increments
    wn_mi_raw = mi_knn(wn_inc[:-1], wn_inc[1:])
    bm_mi_raw = mi_knn(bm_inc[:-1], bm_inc[1:])

    # -------- Library methods (log-energy based) --------
    wn_logE = log_energy_field(wn_inc, s)
    bm_logE = log_energy_field(bm_inc, s)

    wn_cov = log_energy_covariance(wn_logE, max_lag)
    bm_cov = log_energy_covariance(bm_logE, max_lag)

    wn_intvar = intermittency_variance(wn_logE)
    bm_intvar = intermittency_variance(bm_logE)

    # =========================
    # PLOTS
    # =========================

    # 1) Raw signals
    fig1 = make_subplots(rows=2, cols=1,
                         subplot_titles=["White noise", "Brownian motion"])
    fig1.add_trace(go.Scatter(y=wn, mode="lines", name="WN"), row=1, col=1)
    fig1.add_trace(go.Scatter(y=bm, mode="lines", name="BM"), row=2, col=1)
    fig1.update_layout(height=600, title="Raw signals")
    fig1.show()

    # 2) Increment distributions (naive view)
    bins = int(12 * np.sqrt(len(wn_inc)))
    wn_hist, edges = np.histogram(wn_inc, bins=bins, density=True)
    bm_hist, _ = np.histogram(bm_inc, bins=edges, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=centers, y=wn_hist, mode="markers", name="WN increments"))
    fig2.add_trace(go.Scatter(x=centers, y=bm_hist, mode="markers", name="BM increments"))
    fig2.update_layout(title=f"Increment distributions (s={s})")
    fig2.show()

    # 3) Naive metrics comparison (kurtosis, raw variance, raw MI)
    fig3 = make_subplots(rows=1, cols=3,
                         subplot_titles=["Kurtosis (increments)",
                                         "Raw variance (increments)",
                                         "Raw MI (increments)"])

    fig3.add_trace(go.Bar(x=["WN", "BM"], y=[wn_kurt, bm_kurt], name="kurtosis"),
                   row=1, col=1)
    fig3.add_trace(go.Bar(x=["WN", "BM"], y=[wn_var_raw, bm_var_raw], name="variance"),
                   row=1, col=2)
    fig3.add_trace(go.Bar(x=["WN", "BM"], y=[wn_mi_raw, bm_mi_raw], name="MI"),
                   row=1, col=3)

    fig3.update_layout(title="Naive metrics (showing false detection)")
    fig3.show()

    # 4) Log-energy fields (library)
    fig4 = make_subplots(rows=2, cols=1,
                         subplot_titles=["WN log-energy field", "BM log-energy field"])
    fig4.add_trace(go.Scatter(y=wn_logE, mode="lines", name="WN logE"), row=1, col=1)
    fig4.add_trace(go.Scatter(y=bm_logE, mode="lines", name="BM logE"), row=2, col=1)
    fig4.update_layout(height=600, title="Log-energy fields (normalized)")
    fig4.show()

    # 5) Log-energy covariance (true intermittency test)
    fig5 = go.Figure()
    fig5.add_trace(go.Scatter(y=wn_cov, mode="lines", name="WN"))
    fig5.add_trace(go.Scatter(y=bm_cov, mode="lines", name="BM"))
    fig5.update_layout(title="Log-energy covariance (should be ~0 for both)")
    fig5.show()

    # 6) Intermittency variance (library estimator)
    fig6 = go.Figure()
    fig6.add_trace(go.Bar(x=["WN", "BM"], y=[wn_intvar, bm_intvar]))
    fig6.update_layout(title="Intermittency variance (log-energy based)")
    fig6.show()

    # =========================
    # TEXT SUMMARY
    # =========================
    print("\n=== Naive metrics (increments) ===")
    print(f"WN kurtosis: {wn_kurt:.4f}")
    print(f"BM kurtosis: {bm_kurt:.4f}")
    print(f"WN raw variance: {wn_var_raw:.4f}")
    print(f"BM raw variance: {bm_var_raw:.4f}")
    print(f"WN raw MI: {wn_mi_raw:.4f}")
    print(f"BM raw MI: {bm_mi_raw:.4f}")

    print("\n=== Log-energy based metrics ===")
    print(f"WN intermittency variance: {wn_intvar:.4f}")
    print(f"BM intermittency variance: {bm_intvar:.4f}")
    print("WN log-energy covariance (first few lags):", wn_cov[:5])
    print("BM log-energy covariance (first few lags):", bm_cov[:5])
    print("=================================\n")


validate_false_detection()



=== Naive metrics (increments) ===
WN kurtosis: 0.0444
BM kurtosis: -0.1383
WN raw variance: 1.9906
BM raw variance: 61.5880
WN raw MI: -0.2556
BM raw MI: 1.4524

=== Log-energy based metrics ===
WN intermittency variance: 5.1114
BM intermittency variance: 4.9393
WN log-energy covariance (first few lags): [ 0.01999792 -0.00447436  0.10404106  0.09084554  0.04127721]
BM log-energy covariance (first few lags): [3.86017006 3.4140722  3.10676512 2.81857572 2.62455778]

